<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.15 Local Model Context Protocol

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

In [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb) you gave an agent two tools and it cost you a page of hand-written glue: one JSON schema per tool, one dispatch table, one loop, all specific to Ollama's API. If a company runs five AI applications (a chat assistant, a coding agent, a support bot, ...) and wants each to reach eight systems (calendar, CRM, database, file share, ...), scaling is an obvious hardship. Hand-wired, that is $5 \times 8 = 40$ integrations, each one written, secured, and maintained separately.

Wi-Fi ended the era of one radio per vendor pair. USB ended the era of one cable per device. The fix is always the same: **a standard interface**, so that $N \times M$ bespoke connections collapse into $N + M$ standard-compliant ones. For AI tools, that standard now exists: the **Model Context Protocol (MCP)**, released as an open standard by Anthropic in November 2024 [01] and since adopted across the industry, including by OpenAI and Google. An application that speaks MCP can use *any* MCP tool server, and a tool server, written once, serves *any* MCP application.

Today you will not just read about MCP: you will build a real MCP server (in fewer lines than your hand-written glue), speak the protocol to it, and see why your agent code never has to change again when the toolbox does.

<div class="alert alert-block alert-success">

This notebook needs one extra package: run `pip install fastmcp` (or add it via [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb), section 1.2; it is included in the course requirements). The Raspberry Pi is needed only for the optional bridge in Section 5.
</div>

Before any protocol details: the chart below is the entire business case of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N_APPS = 5                                    # AI applications in the company
systems = np.arange(1, 21)                    # systems worth connecting

bespoke = N_APPS * systems                    # every application x every system
standard = N_APPS + systems                   # each side implements the protocol once

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(systems, bespoke, color="#d95f02", linewidth=1.8, marker="o", markersize=4)
ax.plot(systems, standard, color="#1b9e77", linewidth=1.8, marker="o", markersize=4)
ax.fill_between(systems, standard, bespoke, color="#d95f02", alpha=0.07)

ax.annotate("bespoke: N x M", (systems[-1], bespoke[-1]),
            textcoords="offset points", xytext=(-8, 8), ha="right", color="#d95f02")
ax.annotate("standard: N + M", (systems[-1], standard[-1]),
            textcoords="offset points", xytext=(-8, 10), ha="right", color="#1b9e77")
ax.annotate("the gap is the glue code\nnobody has to write", (11, 38),
            ha="center", color="#757575", fontsize=9)

ax.set_xlabel(f"connected systems M   (with N = {N_APPS} AI applications)")
ax.set_ylabel("integrations to build and maintain")
ax.set_xticks(range(2, 21, 2))
ax.set_xlim(1, 20.5)
ax.set_ylim(0, bespoke[-1] * 1.08)
ax.set_title("What a standard interface is worth", loc="left")
plt.show()

At five connected systems the standard has already saved fifteen integrations. At twenty, it saves eighty. Keep this picture in mind: every section below is machinery for moving from the orange line to the green one.

### 2. The Architecture: Host, Client, Server

MCP borrows the shape of every protocol you have studied in this course: well-defined roles, a handshake, then structured messages (JSON-RPC 2.0) over a transport:

```text
+---------------------------+        MCP (JSON-RPC)        +----------------------+
|  HOST  (the AI app)       |   initialize / tools.list    |  SERVER (the tools)  |
|  e.g. Claude, VS Code,    | <==========================> |  e.g. calendar, DB,  |
|  your agent loop          |        tools.call            |  "wireless-lab"      |
|      [ CLIENT ]           |                              +----------------------+
+---------------------------+     transport: stdio (local process)
                                          or HTTP (remote service)
```

- The **host** is the AI application the user sees. Inside it, a **client** maintains one connection per server.
- A **server** typically is a small local program that wraps one system and exposes three kinds of things: **tools** (functions the model may request, like our signal measurement), **resources** (data the application can read, like files or tables), and **prompts** (reusable templates).
- After the handshake, the client asks `tools/list`, so the application *discovers* what is available at runtime instead of having it hard-coded. That discovery step is what makes the toolbox swappable.

### 3. Build a Server, Speak the Protocol

We now rebuild the site-survey tools from [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb), this time as a genuine MCP server. Watch what you do *not* write this time: no JSON schemas, no dispatch table. The `FastMCP` library reads the function signatures and docstrings and generates the schemas itself.

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("wireless-lab")

ROOM_SIGNALS_DBM = {"office": -48, "kitchen": -63, "basement": -81, "garden": -71}

@mcp.tool
def list_rooms() -> list[str]:
    """List the rooms of the building."""
    return list(ROOM_SIGNALS_DBM)

@mcp.tool
def measure_wifi_signal(room: str) -> str:
    """Measure the current Wi-Fi signal strength in one room, in dBm."""
    return f"{ROOM_SIGNALS_DBM.get(room.lower().strip(), -95)} dBm"

print(f"MCP server {mcp.name!r} is defined.")
print("No schemas were written - they will be generated from the type hints and docstrings.")

That is the entire server. Now the other side: we open a client session against it and hold a real MCP conversation: handshake, discovery, call. (The client here connects *in memory*, which is perfect for inspection; to real applications the same server would talk over stdio or HTTP, with not one line of its code changed.)

In [ ]:
from fastmcp import Client

async with Client(mcp) as session:
    tools = await session.list_tools()                       # -> JSON-RPC: tools/list
    print(f"the client discovered {len(tools)} tools:\n")
    for tool in tools:
        print(f"  {tool.name}")
        print(f"    description : {tool.description}")
        print(f"    schema      : {tool.inputSchema}\n")

    result = await session.call_tool(                        # -> JSON-RPC: tools/call
        "measure_wifi_signal", {"room": "kitchen"})
    print(f"tools/call measure_wifi_signal(room='kitchen') -> {result.content[0].text}")

You just spoke MCP. Two details deserve a second look. The schemas the client received are the ones you *did not write*. Compare them with the hand-made ones in [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb): they carry the same information, generated from the Python signature. And the tool list was *discovered*, not assumed. Add a third tool to the server cell, rerun both cells, and the client finds it with no other change anywhere.

Under the hood, the session exchanged ordinary JSON-RPC messages, the same style of request/response framing you have seen in every protocol of this course. Illustratively:

```json
-> {"jsonrpc": "2.0", "id": 2, "method": "tools/list"}
<- {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"name": "measure_wifi_signal", ...}]}}

-> {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
    "params": {"name": "measure_wifi_signal", "arguments": {"room": "kitchen"}}}
<- {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "-63 dBm"}]}}
```

### 4. The Same Server, Any Host

Here is where the standard pays out. To offer your `wireless-lab` server to a real AI application, you save the server cell as a file (plus two closing lines) and *register* it. No code, just configuration:

```python
# wireless_lab_server.py - the cell from Section 3, plus:
if __name__ == "__main__":
    mcp.run()          # serve over stdio
```

```json
// Claude Desktop - claude_desktop_config.json
{"mcpServers": {"wireless-lab": {"command": "python", "args": ["wireless_lab_server.py"]}}}
```

```json
// VS Code - .vscode/mcp.json
{"servers": {"wireless-lab": {"type": "stdio", "command": "python", "args": ["wireless_lab_server.py"]}}}
```

One server, written once, usable from Claude Desktop, VS Code, Cursor, and any other MCP host, including your own agent. That is $N + M$ in practice, and it is why an ecosystem of thousands of ready-made servers (GitHub, Slack, databases, file systems, browsers, ...) appeared within months of the standard's release.

### 5. Optional: Your Agent Meets the Standard

If your Raspberry Pi is connected (tunnel from [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb), Section 2), run the closing experiment: the agent loop from Local Agents I, except that its toolbox now comes from the MCP server: discovered at runtime, executed through the protocol. Note that nothing had to change in the agent logic. The tools became universal and the loop did not notice.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"       # the Pi, through the SSH tunnel
AGENT_MODEL = "qwen3:1.7b"

try:
    requests.get(f"{OLLAMA_URL}/api/version", timeout=3)
    PI_ONLINE = True
except requests.exceptions.RequestException:
    PI_ONLINE = False

async def run_agent_with_mcp(task, max_steps=6):
    async with Client(mcp) as session:
        # 1. discover the toolbox through the protocol and hand it to the model
        discovered = await session.list_tools()
        tool_schemas = [{"type": "function", "function": {
            "name": tool.name, "description": tool.description or "",
            "parameters": tool.inputSchema}} for tool in discovered]

        messages = [
            {"role": "system", "content": "You are a facilities assistant. Gather evidence "
             "with the available tools before answering. Two sentences maximum."},
            {"role": "user", "content": task},
        ]
        # 2. the exact same loop as in Local Agents I
        for step in range(1, max_steps + 1):
            reply = requests.post(f"{OLLAMA_URL}/api/chat", json={
                "model": AGENT_MODEL, "messages": messages, "tools": tool_schemas,
                "stream": False, "think": False,
                "options": {"temperature": 0.2, "seed": 7}}, timeout=600).json()["message"]
            messages.append(reply)
            if not reply.get("tool_calls"):
                print(f"[step {step}] ANSWER: {reply['content'].strip()}")
                return
            for call in reply["tool_calls"]:
                name = call["function"]["name"]
                arguments = call["function"].get("arguments") or {}
                if isinstance(arguments, str):
                    import json
                    arguments = json.loads(arguments)
                try:
                    outcome = await session.call_tool(name, arguments)   # 3. execute via MCP
                    result = outcome.content[0].text
                except Exception as error:
                    result = f"tool error: {error}"
                print(f"[step {step}] TOOL   {name}({arguments}) -> {result}")
                messages.append({"role": "tool", "content": result, "tool_name": name})
        print(f"[stopped] step budget of {max_steps} reached.")

if PI_ONLINE:
    await run_agent_with_mcp("Which room is best for an important video call? "
                             "Check the signal in every room before you decide.")
else:
    print("❌ Pi not connected - this closing demo needs the tunnel from Local Agents I.")

### 6. The Manager's View: Standards, Again

Strip away the acronym and MCP is a story you already know how to evaluate, because this course keeps telling it: open standard versus proprietary integration, ecosystem versus lock-in, $N + M$ versus $N \times M$. The strategic questions are the same ones asked about Wi-Fi and USB in their day: who governs the standard, how broad is adoption, what happens to your bespoke investments, and the history of those earlier standards is a fair guide to the likely answer.

One question, however, is new and sharper here, and it should be on every checklist before an MCP server is enabled in a company: **a tool call is an action**. Every server you plug in extends what an AI application can *do* (read your files, query your database, send messages) on the strength of text produced by a model. Before enabling one, ask what you would ask of any new employee's system access: which permissions exactly, whose credentials, what is logged, and which actions require a human's confirmation. The protocol standardizes the plug but the responsibility for what flows through it stays with you.

### Further Reading and Literature

[01] Anthropic (2024). **Introducing the Model Context Protocol.** Announcement, November 2024. [Article](https://www.anthropic.com/news/model-context-protocol)

[02] **Model Context Protocol: specification and documentation.** The standard itself, including the tools/resources/prompts primitives and transports. [Website](https://modelcontextprotocol.io)

[03] **FastMCP documentation.** The Python library used in this notebook. [Website](https://gofastmcp.com)

[04] Ollama (2024). **Tool support.** The model-side half of the bridge in Section 5. [Blog](https://ollama.com/blog/tool-support)

----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**From your run of Section 3: how many tools did the client discover, and what does the `inputSchema` of `measure_wifi_signal` contain? Where did that schema come from, given that you never wrote it?** Answer:

(add your answer here)

**Explain host, client, and server in your own words, and map each role onto what happened in the in-memory session of Section 3.** Answer:

(add your answer here)

**Add a third tool to the server (for example `count_devices(room)` returning an invented number), rerun the two cells, and paste the new discovery output. What did you have to change on the client side?** Answer:

(add your answer here)

**Redraw the Section 1 chart for your own organization: set `N_APPS` and the range of `systems` to realistic estimates for your company, rerun the cell, and report the two integration counts at your M. At what N and M does the standard start to win?** Answer:

(add your answer here)

**If you ran the Section 5 bridge: paste the trace and identify which parts came through MCP (discovery, execution) and which through Ollama (the decisions). If you could not run it, explain the division using the code.** Answer:

(add your answer here)

**The Wi-Fi/USB analogy for MCP is helpful but imperfect. Name one way it fits well and one way it breaks down (hint: Section 6. What does a tool call do that a data packet does not?).** Answer:

(add your answer here)

**Your team wants to enable a community-built MCP server for the company CRM. Write the three questions you would require answered before approving it, and say what evidence would satisfy each.** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)